### 2D Integrals with Taylor expansions

Any definite and proper 2-dimensional integral can be discretized as shown:

$$
    \int_{a}^{b} \int_{c}^{d} \, f(x, y) \, dxdy = \sum_{i = 0}^{N_x - 1} \sum_{j = 0}^{N_y - 1} \, \int_{x_i}^{x_i + h_x} \int_{y_j}^{y_j + h_y} \, f(x, y) \, dxdy
$$

With $x_0 = a$, $x_{Nx} = b$, $h_x = \frac{b - a}{N_x}$ and $y_0 = c$, $y_{Ny} = d$, $h_y = \frac{d - c}{N_y}$

Trick: expand the function $f(x, y)$ in a Taylor series about the point $x_i + \frac{h_x}{2}, y_j + \frac{h_y}{2}$ and integrate in $\left[ \left( x_i, x_i + h_x \right), \left( y_j, y_j + h_y \right) \right]$

Using Maxima CAS we can write this approximation formula:

$$
    \int_{a}^{b} \int_{c}^{d} \, f(x, y) \, dxdy \approx h_x h_y \, \sum_{i = 0}^{N_x - 1} \sum_{j = 0}^{N_y - 1} \, f \left( x_i + \frac{h_x}{2}, y_j + \frac{h_y}{2} \right)
$$

The upper bound of the leading-order truncation error is given by:

$$
    E \leq \frac{h_x ^ 2}{24} \int_c ^ d \left| f_x (b, y) - f_x (a, y) \right| \, dy + \frac{h_y ^ 2}{24}\int_a ^ b \left| f_y (x, d) - f_y (x, c) \right| \, dx
$$

In [1]:
import numpy as np
from scipy.integrate import dblquad

In [2]:
f = lambda x, y: np.exp(- x ** 2 - y ** 4) # f(x, y) = e^{-x^2 - y^4}
df_dx = lambda x, y: - 2.0 * x * f(x, y) # derivative of f(x, y) with respect of x
df_dy = lambda x, y: - 4.0 * y ** 3 * f(x, y) # derivative of f(x, y) with respect of y

In [3]:
def integral_2D(f, df_dx, df_dy, ab, cd, hx, hy): # Implements the formula above

    def _integral_1D(g, xa, xb, h): # 1D equivalent
        ba = xb - xa
        N = round(ba / h)
        h = ba / N
        x = np.linspace(xa, xb, N, endpoint = False)
        midpoints = x + h / 2.0
        return h * np.sum(g(midpoints))

    # Ranges, subdivisions and steps
    a, b = ab
    c, d = cd
    Nx = round((b - a) / hx)
    Ny = round((d - c) / hy)
    hx = (b - a) / Nx
    hy = (d - c) / Ny

    # Midpoints
    xs = a + (np.arange(Nx) + 0.5) * hx
    ys = c + (np.arange(Ny) + 0.5) * hy

    X, Y = np.meshgrid(xs, ys, indexing = "ij") # Meshgrid

    I = hx * hy * np.sum(f(X, Y)) # 2D integral result

    term_x = (hx ** 2) * _integral_1D( # First term of error formula
        lambda y: np.abs(df_dx(b, y) - df_dx(a, y)),
        c, d, hy
    )

    term_y = (hy ** 2) * _integral_1D( # Second term of the error formula
        lambda x: np.abs(df_dy(x, d) - df_dy(x, c)),
        a, b, hx
    )

    err = (term_x + term_y) / 24.0 # Computing the upper bound

    return I, err # 2D Integral and the upper bound error

In [4]:
ab = (- 3.0, 3.0) # a = -3, b = 3
cd = (- 3.0, 3.0) # c = -3, d = 3
hx, hy = 0.005, 0.005 #  Same step

res = integral_2D(f, df_dx, df_dy, ab, cd, hx, hy) # Result
print(f"{res[0]:.9f} +- {res[1]:.9f}")

3.213042145 +- 0.000000003


In [5]:
res_scipy = dblquad(f, ab[0], ab[1], cd[0], cd[1]) # from Scipy
print(f"{res_scipy[0]:.8f} +- {res_scipy[1]:.8f}")

3.21304214 +- 0.00000003
